# Lab 6 — Incremental File Ingestion with Auto Loader (`cloudFiles`)

**Goal:** Create JSON files, ingest them incrementally using Auto Loader, and write to a Bronze Delta table.

**Prerequisites:** Databricks compute with Auto Loader support and a writable Unity Catalog volume or cloud-storage path. The `/tmp` paths from the source guide are not reliable across clusters, so this notebook uses a configurable path. Set `BASE_PATH` to a location your compute can write to, preferably a Unity Catalog Volume such as `/Volumes/<catalog>/<schema>/<volume>/lab06`.

In [0]:
BASE_PATH = "/Volumes/workspace/lab_db/lab06_files"

LANDING_PATH = f"{BASE_PATH}/landing/orders"
SCHEMA_PATH = f"{BASE_PATH}/schema"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/orders"

BRONZE_TABLE = "workspace.lab_db.bronze_orders_lab6"

print("Landing:", LANDING_PATH)
print("Schema:", SCHEMA_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

## Create sample JSON landing files

This creates small sample files for practice. On some serverless or governed configurations, writing to `/tmp` may not be supported; use a Unity Catalog Volume path instead.

In [0]:
import json, os
from datetime import datetime, timezone

os.makedirs(LANDING_PATH, exist_ok=True)
sample_orders = [
    {"order_id": "O-1001", "customer_id": 101, "amount": 125.50, "order_timestamp": "2026-10-12T09:00:00Z"},
    {"order_id": "O-1002", "customer_id": 102, "amount": 75.00, "order_timestamp": "2026-10-12T09:02:00Z"},
    {"order_id": "O-1003", "customer_id": 103, "amount": 240.00, "order_timestamp": "2026-10-12T09:05:00Z"}
]
for i, record in enumerate(sample_orders, start=1):
    path = os.path.join(LANDING_PATH, f"orders_{i:03d}.json")
    with open(path, "w") as f:
        json.dump(record, f)
print(f"Wrote {len(sample_orders)} sample files to {LANDING_PATH}")

In [0]:
# Auto Loader read and write
from pyspark.sql.functions import col

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .load(LANDING_PATH))

query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE))

query.awaitTermination()
print("Auto Loader run completed.")

In [0]:
display(spark.table(BRONZE_TABLE).orderBy("order_id"))

## Verification

Expected: at least 3 sample order rows. For a rerun, add new files with new filenames; Auto Loader checkpoints track files already processed.

In [0]:
assert spark.table(BRONZE_TABLE).count() >= 3, "Expected at least 3 ingested orders"
print("PASS: Bronze table contains the sample orders.")